<a href="https://colab.research.google.com/github/maddy-aleti/AgencyProject/blob/main/road_reconstruction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch

print("pytorch: ",torch.__version__)
print("CUDA available: ",torch.cuda.is_available())

if torch.cuda.is_available():
   print("GPU: ",torch.cuda.get_device_name(0))
else:
    print("Enable a GPU runtime in Colab before training")

pytorch:  2.11.0+cu130
CUDA available:  True
GPU:  Tesla T4


In [4]:
%pip install -q segmentation-models-pytorch albumentations opencv-python-headless matplotlib scikit-learn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.8 MB/s eta 0:00:00


In [ ]:
import zipfile
from huggingface_hub import hf_hub_download
from pathlib import Path
import os

DATA_DIR =Path("/content/deepglobe")
DATA_DIR.mkdir(parents=True,exist_ok=True)


zip_path = hf_hub_download(
    repo_id="k4nnguyen/deepglobe-roads",
    filename="datasetdg.zip",
    repo_type="dataset"
)

print("Download Archive: ",zip_path)

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(DATA_DIR)

print("Extraction complete:",DATA_DIR)


datasetdg.zip: reconstructing file:   0%|          |  0.00B / 2.96GB            

datasetdg.zip: downloading bytes:           |  0.00B            

Download Archive:  /root/.cache/huggingface/hub/datasets--k4nnguyen--deepglobe-roads/snapshots/f9a0397ef21b59c5f198b7a274b1f469c1e7e906/datasetdg.zip
Extraction complete: /content/deepglobe


In [2]:
from pathlib import Path

all_files= list(DATA_DIR.rglob("*"))

sat_files = [
    p for p in all_files
    if p.is_file() and p.name.endswith("_sat.jpg")
]

mask_files= [
    p for p in all_files
    if p.is_file() and p.name.endswith("_mask.png")
]

print("Number of satellite images: ",len(sat_files))
print("Number of mask images: ",len(mask_files))

print("\nExample images:")
for p in sat_files[:5]:
  print(p)

print("\nExample masks: ")
for p in mask_files[:5]:
  print(p)


NameError: name 'DATA_DIR' is not defined

In [ ]:
mask_index= {p.name : p for p in mask_files}

pairs=[]

for image_path in sat_files:
  mask_name = image_path.name.replace(
      "_sat.jpg","_mask.png"
  )

  mask_path =mask_index.get(mask_name)

  if mask_path is not None:
     pairs.append((image_path,mask_path))

print("Valid image-mask pairs: ",len(pairs))

assert len(pairs)>0,(
       "No image-mask pairs found .Inspect the extracted folder"
       "and adjust the filename matching logic"
)

print("First Pair")
print("Image:", pairs[0][0])
print("Mask: ",pairs[0][1])



Valid image-mask pairs:  6226
First Pair
Image: /content/deepglobe/datasetdg/ROAD/training/images/649332_sat.jpg
Mask:  /content/deepglobe/datasetdg/ROAD/training/masks/649332_mask.png


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def read_pair(image_path,mask_path,size=None):
    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)

    mask= cv2.imread(str(mask_path),cv2.IMREAD_GRAYSCALE)

    if size is not None:
       image =cv2.resize(
              image,(size,size),
              interpolation =cv2.INTER_LINEAR
       )
       mask =cv2.resize(
           mask,(size,size),
           interpolation=cv2.INTER_NEAREST
       )

    mask=(mask>127).astype(np.uint8)
    return image,mask
image,mask= read_pair(*pairs[0])

fig,axes =plt.subplots(1,3,figsize=(15,5))

axes[0].imshow(image)
axes[0].set_title("Satellite image")

axes[1].imshow(mask,cmap="gray",vmin=0,vmax=1)
axes[1].set_title("Ground-truth road mask")

axes[2].imshow(image)
axes[2].imshow(mask,cmap="autumn",alpha=0.5)
axes[2].set_title("Road mask overlay")

for ax in axes:
  ax.axis("off")

plt.tight_layout()
plt.show()

print("Image shape:",image.shape)
print("Mask shape: ",mask.shape)
print("Road pixel fraction: ",mask.mean())

NameError: name 'pairs' is not defined

In [ ]:
from sklearn.model_selection import train_test_split

SEED=42

train_pairs,temp_pairs= train_test_split(
    pairs,
    test_size=0.20,
    random_state=SEED
)

val_pairs,test_pairs =train_test_split(
    temp_pairs,
    test_size=0.50,
    random_state=SEED
)

print(f"Train: {len(train_pairs)}")
print(f"Validation: {len(val_pairs)}")
print(f"Test: {len(test_pairs)}")

Train: 4980
Validation: 623
Test: 623


In [ ]:
import random
import torch
from torch.utils.data import Dataset,DataLoader
from torch.utils.data import default_collate
import numpy as np

IMAGE_SIZE = 512
IMAGENET_MEAN = np.array(
    [0.485,0.456,0.406],dtype=np.float32
    ).reshape(1,1,3)

IMAGENET_STD = np.array(
    [0.229,0.224,0.225],dtype=np.float32
).reshape(1,1,3)

class RoadDataset(Dataset):
  def __init__(self,pairs,image_size=512,augment=False):
      self.pairs = pairs
      self.image_size =image_size
      self.augment= augment

  def __len__(self):
      return len(self.pairs)

  def __getitem__(self,idx):
    image_path,mask_path =self.pairs[idx]

    image,mask= read_pair(
        image_path,mask_path,self.image_size
    )

    if self.augment:
       if random.random() < 0.5:
         image = np.fliplr(image).copy()
         mask = np.fliplr(mask).copy()

       if random.random() <0.5:
          image=np.flipud(image).copy()
          mask = np.flipud(mask).copy()

       k=random.randint(0,3)
       image= np.rot90(image, k).copy()
       mask =np.rot90(mask, k).copy()

    image = image.astype(np.float32) /255.0
    image = (image - IMAGENET_MEAN) /IMAGENET_STD

    image= torch.from_numpy(
           image.transpose(2,0,1).copy()
    ).float()

    mask = torch.from_numpy(mask).unsqueeze(0).float()

    return image,mask

train_ds=RoadDataset(
    train_pairs,IMAGE_SIZE,augment=True
)

val_ds =RoadDataset(
    val_pairs,IMAGE_SIZE,augment=False
)

test_ds =RoadDataset(
    test_pairs,IMAGE_SIZE,augment=False
)

BATCH_SIZE = 4
NUM_WORKERS =2

train_loader= DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False
)

val_loader =DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader =DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

images, masks= next(iter(train_loader))

print("Image batch: ",images.shape)
print("Mask batch: ",masks.shape)

NameError: name 'train_pairs' is not defined